# 파일럿 이어서 실행 (Kaggle, P0K · P3K · P4~P7)

**시작 전 설정 (오른쪽 패널 Session options)**
1. Accelerator: **GPU T4 x2**
2. Internet: **On** (전화번호 인증 필요)
3. Input: 코랩 결과를 올린 데이터셋(예: `hn-runs-colab`) 추가

**실행 방법**: 오른쪽 위 **Save Version → Save & Run All (Commit)**. 브라우저를 닫아도 끝까지 돌고, 결과는 Output에 저장됩니다.
결과는 모두 로컬 디스크(`/kaggle/working/runs`)에 저장되므로 구글 드라이브 때문에 멈추는 일이 없습니다.

## 1. 환경 변수 + 코드 불러오기 + GPU 확인

In [ ]:
import os, subprocess
BRANCH = 'itkyoung0606-pilot-ing'
REPO = '/tmp/hard-negative-mllm'          # 코드는 Output에 섞이지 않게 /tmp에
RUNS = '/kaggle/working/runs'             # 결과는 Output으로 저장되는 곳
os.environ.update({'HN_RUNS_DIR': RUNS, 'HN_N_GPU': '2', 'USE_TF': '0',
                   'TF_CPP_MIN_LOG_LEVEL': '3', 'TOKENIZERS_PARALLELISM': 'false'})
os.makedirs(RUNS, exist_ok=True)

if not os.path.exists(REPO):
    !git clone -q -b {BRANCH} https://github.com/naha0106-tech/hard-negative-mllm.git {REPO}
%cd {REPO}
!git fetch -q origin {BRANCH}
!git reset -q --hard origin/{BRANCH}
!git log --oneline -1
!nvidia-smi --query-gpu=index,name,memory.total --format=csv,noheader
gpus = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout
assert gpus.count('Tesla T4') >= 1, 'T4 GPU가 아닙니다. Accelerator를 GPU T4 x2로 바꾸세요.'
os.environ['HN_N_GPU'] = str(gpus.count('Tesla T4'))
print('사용할 T4 개수 =', os.environ['HN_N_GPU'])

## 2. 설치 + 버전 기록

In [ ]:
!pip install -q transformers==4.51.3 peft==0.15.2 accelerate datasets pyarrow pyyaml 2>&1 | grep -v "dependency resolver\|requires\|incompatible" || true
!python -c "import sys; print('python', sys.version.split()[0])" | tee {RUNS}/env_kaggle.txt
!pip freeze | grep -E "^(torch|transformers|peft|accelerate|datasets)==" | tee -a {RUNS}/env_kaggle.txt
import yaml
print('SmolVLM revision =', yaml.safe_load(open('config.yaml'))['models']['smolvlm']['revision'])
print('학습률 =', yaml.safe_load(open('config.yaml'))['train']['lr'])

## 3. 코랩 결과(P0, P1~P3) 가져오기
Input 데이터셋에서 `pilot_lr.json`이 있는 폴더를 찾아 그 안의 run 폴더들을 `/kaggle/working/runs`로 복사합니다.

In [ ]:
import glob, shutil, json
found = glob.glob('/kaggle/input/**/pilot_lr.json', recursive=True)
assert found, 'Input에 코랩 결과(pilot_lr.json이 든 runs 폴더)가 없습니다. 데이터셋을 추가하세요.'
src = os.path.dirname(found[0])
print('가져올 위치:', src)
for name in sorted(os.listdir(src)):
    s, d = os.path.join(src, name), os.path.join(RUNS, name)
    if os.path.exists(d):
        continue
    (shutil.copytree if os.path.isdir(s) else shutil.copy2)(s, d)
    print('  복사:', name)
print('채택 학습률(코랩) =', json.load(open(f'{RUNS}/pilot_lr.json'))['chosen_lr'])
for rid in ['pilot_P0_ZS', 'pilot_P1_Rand_N400_lr5e-5', 'pilot_P2_Rand_N400_lr1e-4', 'pilot_P3_Rand_N400_lr2e-4']:
    assert os.path.exists(f'{RUNS}/{rid}/DONE'), f'{rid}가 없습니다'
print('코랩 결과 확인 완료')

## 4. P0K — Kaggle 환경에서 학습 전 점검 C1~C8
코랩과 같은 결과(C1 = 0.640 근처)가 나오는지 확인합니다. 실패하면 여기서 멈춥니다.

In [ ]:
!python scripts/04a_pilot_checks.py --run_id pilot_P0K_ZS
assert os.path.exists(f'{RUNS}/pilot_P0K_ZS/DONE'), 'P0K 점검 실패 — 위 출력에서 실패 줄을 확인하세요'

## 5. P4~P7 + P3K 학습 (GPU 2개 동시)
(P4 Rand, P5 Hard) → (P6 Easy, P7 Rand 반복) → P3K(코랩 P3을 Kaggle에서 다시). 각 학습의 자세한 출력은 `runs/_logs/`에 있습니다.

In [ ]:
!python scripts/run_queue.py --pilot

## 6. 관문 B 성적표

In [ ]:
!python scripts/04b_pilot_report.py

## 7. 결과 정리 (요약 표 + 내려받기용 zip)

In [ ]:
import pandas as pd
rows = []
for p in sorted(glob.glob(f'{RUNS}/pilot_*/summary.json')):
    s = json.load(open(p)); s['run'] = os.path.basename(os.path.dirname(p)); rows.append(s)
display(pd.DataFrame(rows).set_index('run'))
shutil.make_archive('/kaggle/working/pilot_results', 'zip', RUNS)
print('내려받기: Output → pilot_results.zip')